# Can we predict 30-day readmission?
Data: `data.csv` (one row per hospital stay), dictionary in `columns.csv`. Target: `readmit_30d`.

Key design point: rows are **repeated stays of the same person** (`person_key`). A random row split would put the same person in train and test, so every evaluation here is **grouped by person**. `person_key` itself is an identifier and is never used as a feature.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width',140)
df=pd.read_csv('data.csv'); print(pd.read_csv('columns.csv').to_string(index=False))
df.shape

## 1. Data checks

In [ ]:
print(df.isna().sum().sum(),'missing;',df.duplicated().sum(),'duplicate rows;',df.duplicated(['person_key','visit_number']).sum(),'duplicate person+visit')
print(df.person_key.nunique(),'people, stays per person:',df.groupby('person_key').size().describe()[['min','mean','max']].round(1).to_dict())
print('readmission rate: %.3f'%df.readmit_30d.mean())
df.describe().T.round(2)

## 2. Univariate look: readmission rate by feature

In [ ]:
num=['visit_number','birth_year','postcode_area','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
fig,ax=plt.subplots(2,4,figsize=(16,7))
for a,c in zip(ax.ravel(),num):
    q=pd.qcut(df[c],10,duplicates='drop'); r=df.groupby(q,observed=True).readmit_30d.mean()
    a.plot(range(len(r)),r.values,'o-'); a.axhline(df.readmit_30d.mean(),ls='--',c='grey'); a.set_title(c); a.set_xlabel('decile'); a.set_ylabel('readmit rate')
plt.tight_layout(); plt.show()
print(df.groupby('sex').readmit_30d.mean())

## 3. Models (5-fold, grouped by person)
Baselines: a no-skill dummy and logistic regression; then gradient boosting to capture non-linearity/interactions. Out-of-fold predictions are pooled for metrics.

In [ ]:
df['sex_M']=(df.sex=='M').astype(int)  # numeric encoding so all models can use it
feats=num+['sex_M']; X=df[feats]; y=df.readmit_30d; g=df.person_key
models={'dummy':DummyClassifier(strategy='prior'),
 'logreg':make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)),
 'hist_gb':HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,l2_regularization=1.0,random_state=0)}
cv=GroupKFold(n_splits=5); oof={}; rows=[]
for n,m in models.items():
    oof[n]=cross_val_predict(m,X,y,groups=g,cv=cv,method='predict_proba')[:,1]
    rows.append((n,roc_auc_score(y,oof[n]),average_precision_score(y,oof[n]),brier_score_loss(y,oof[n])))
res=pd.DataFrame(rows,columns=['model','ROC AUC','PR AUC (base=%.2f)'%y.mean(),'Brier']).set_index('model'); res.round(3)

Uncertainty: bootstrap over **people** (not rows) for the AUC of each model.

In [ ]:
rng=np.random.default_rng(0); ppl=df.person_key.unique(); idx=df.groupby('person_key').indices
def boot(p,B=300):
    out=[]
    for _ in range(B):
        s=np.concatenate([idx[k] for k in rng.choice(ppl,len(ppl))]); out.append(roc_auc_score(y.iloc[s],p[s]))
    return np.percentile(out,[2.5,97.5])
for n in ['logreg','hist_gb']: print(n,'AUC 95%% CI: %.3f-%.3f'%tuple(boot(oof[n])))

In [ ]:
for n in ['logreg','hist_gb']:
    f,t,_=roc_curve(y,oof[n]); plt.plot(f,t,label=f'{n} (AUC {roc_auc_score(y,oof[n]):.3f})')
plt.plot([0,1],[0,1],'k--'); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.legend(); plt.title('ROC, grouped CV'); plt.show()
pt,pp=calibration_curve(y,oof['logreg'],n_bins=10,strategy='quantile')
plt.plot(pp,pt,'o-'); plt.plot([0,.6],[0,.6],'k--'); plt.xlabel('predicted'); plt.ylabel('observed'); plt.title('Calibration (logreg)'); plt.show()

## 4. What drives the prediction?

In [ ]:
lr=models['logreg'].fit(X,y)
coef=pd.Series(lr[-1].coef_[0],index=feats).sort_values(); print('Standardised logistic coefficients:'); print(coef.round(3))
imp=np.zeros(len(feats))
for tr,te in cv.split(X,y,g):
    m=models['hist_gb'].fit(X.iloc[tr],y.iloc[tr])
    imp+=permutation_importance(m,X.iloc[te],y.iloc[te],scoring='roc_auc',n_repeats=5,random_state=0).importances_mean/5
pd.Series(imp,index=feats).sort_values().plot.barh(title='Permutation importance (AUC drop, held-out)'); plt.show()

## 5. Practical value: lift when flagging the highest-risk stays

In [ ]:
p=oof['logreg']; d=pd.DataFrame({'p':p,'y':y}); d['decile']=pd.qcut(d.p.rank(method='first'),10,labels=False)+1
t=d.groupby('decile').y.agg(['mean','size']).rename(columns={'mean':'observed rate'}); t['lift']=t['observed rate']/y.mean(); t.round(3)

## 6. Conclusions (from a grouped 5-fold CV run)
- **Weak but real signal.** Logistic regression: ROC AUC ≈ 0.65 (bootstrap 95% CI ≈ 0.63-0.66), PR AUC ≈ 0.32 vs a 0.20 base rate. Gradient boosting is no better (≈ 0.63), so there is no evidence of strong non-linear structure.
- **Drivers:** length of stay, number of medications and lab A (higher → more readmission). Other features add little; `sex`, `lab_b`, `visit_number`, `birth_year` are ≈ noise.
- **Usable for ranking, not for individual decisions.** The top risk decile is readmitted ~39% of the time (≈ 2x lift) and the bottom ~10%, but the model cannot separate individuals confidently.
- **Caveats:** the dummy's AUC (0.47) is just per-fold prior noise. There are no diagnosis, prior-utilisation or timing fields, readmissions elsewhere are not captured, and this is internal CV only (no temporal/external validation). Metrics were computed on rows pooled across folds with people kept intact within folds.